In [ ]:
# 1. Load All Data Sources
import pandas as pd
import json
from pathlib import Path

# File paths
queue_csv = Path('reports/model_feedback_queue.csv')
queue_json = Path('reports/model_feedback_queue.json')
trend_csv = Path('reports/reconciliation_trend_report.csv')
health_json = Path('ledger/ledger_health_snapshot.json')
calib_patch_json = Path('learning/calibration_patch_v1.json')

# Load feedback queue
queue_df = pd.read_csv(queue_csv) if queue_csv.exists() else pd.DataFrame()
with open(queue_json, 'r', encoding='utf-8') as f:
    queue_data = json.load(f) if queue_json.exists() else []

# Load trend report
trend_df = pd.read_csv(trend_csv) if trend_csv.exists() else pd.DataFrame()

# Load ledger health snapshot
with open(health_json, 'r', encoding='utf-8') as f:
    health_data = json.load(f) if health_json.exists() else {}

# Load calibration patch
with open(calib_patch_json, 'r', encoding='utf-8') as f:
    calib_patch = json.load(f) if calib_patch_json.exists() else {}

print('Loaded:')
print(f'- Feedback queue: {len(queue_df)} items')
print(f'- Trend report: {len(trend_df)} rows')
print(f'- Ledger health: {len(health_data) if isinstance(health_data, dict) else 'N/A'}')
print(f'- Calibration patch: {len(calib_patch) if isinstance(calib_patch, dict) else 'N/A'}')

# AI-RISA Model Feedback Dashboard

This dashboard surfaces the highest-priority model feedback targets, links them to trend and calibration context, and enables targeted review for model improvement.

**Data sources:**
- Model feedback queue (CSV/JSON)
- Reconciliation trend report
- Ledger health snapshot
- Calibration patch recommendations

**Sections:**
1. Load all data sources
2. Display top feedback queue items
3. Breakdown of queue by issue type
4. Link feedback items to trend context
5. Show calibration patch recommendations
6. Add filters for family, version, and issue type

## 2. Display Top Feedback Queue Items

This section shows the highest-priority feedback targets, their ranking reasons, affected prediction families, model/calibration versions, suggested actions, and relevant metrics.

In [ ]:
# Display top N feedback queue items
from IPython.display import display, Markdown

TOP_N = 10
if not queue_df.empty:
    display(Markdown(f'### Top {min(TOP_N, len(queue_df))} Feedback Targets'))
    display(queue_df.head(TOP_N))
else:
    display(Markdown('No feedback queue items found.'))

## 3. Breakdown of Queue by Issue Type

This section aggregates and visualizes the feedback queue by issue type, showing counts and proportions for each type.

In [ ]:
# Breakdown of queue by issue type
import matplotlib.pyplot as plt

if not queue_df.empty and 'type' in queue_df.columns:
    type_counts = queue_df['type'].value_counts()
    display(Markdown('### Feedback Queue Breakdown by Issue Type'))
    display(type_counts)
    type_counts.plot(kind='bar', color='skyblue', title='Feedback Queue by Issue Type')
    plt.ylabel('Count')
    plt.xlabel('Issue Type')
    plt.show()
else:
    display(Markdown('No issue type data available in feedback queue.'))

## 4. Link Feedback Items to Trend Context

For each feedback item, this section shows relevant trend data and ledger health metrics to provide actionable context.

In [ ]:
# Link feedback items to trend context
if not queue_df.empty and not trend_df.empty:
    display(Markdown('### Feedback Items with Trend Context'))
    for idx, row in queue_df.iterrows():
        context = []
        # Try to link by family_id or version if present
        fam = row.get('family_id') if 'family_id' in row else None
        ver = row.get('version') if 'version' in row else None
        display(Markdown(f"**Target {idx+1}:** {row.get('type', '')} | {row.get('reason', '')}"))
        if fam and 'prediction_family_id' in trend_df.columns:
            fam_trend = trend_df[trend_df['prediction_family_id'] == fam]
            if not fam_trend.empty:
                display(Markdown(f"Trend rows for family `{fam}`:"))
                display(fam_trend.head(3))
        if ver and 'model_version' in trend_df.columns:
            ver_trend = trend_df[trend_df['model_version'] == ver]
            if not ver_trend.empty:
                display(Markdown(f"Trend rows for model version `{ver}`:"))
                display(ver_trend.head(3))
        # Show winner/method/round metrics from health snapshot
        if isinstance(health_data, dict):
            metrics = {k: v for k, v in health_data.items() if 'accuracy' in k or 'round' in k}
            if metrics:
                display(Markdown('**Current Metrics:**'))
                display(metrics)
else:
    display(Markdown('No trend or queue data available for context linking.'))

## 5. Show Calibration Patch Recommendations

This section displays calibration patch recommendations and links them to affected feedback items.

In [ ]:
# Show calibration patch recommendations
if isinstance(calib_patch, dict) and 'recommended_actions' in calib_patch:
    display(Markdown('### Calibration Patch Recommendations'))
    for action in calib_patch['recommended_actions']:
        display(Markdown(f'- {action}'))
else:
    display(Markdown('No calibration patch recommendations found.'))

## 6. Add Filters for Family, Version, and Issue Type

Use the filters below to focus the review on specific prediction families, model/calibration versions, or issue types.

In [ ]:
# Interactive filters for family, version, and issue type
import ipywidgets as widgets
from IPython.display import clear_output

if not queue_df.empty:
    fams = sorted(queue_df['family_id'].dropna().unique()) if 'family_id' in queue_df.columns else []
    vers = sorted(queue_df['version'].dropna().unique()) if 'version' in queue_df.columns else []
    types = sorted(queue_df['type'].dropna().unique()) if 'type' in queue_df.columns else []

    fam_dd = widgets.Dropdown(options=['All'] + fams, description='Family:') if fams else None
    ver_dd = widgets.Dropdown(options=['All'] + vers, description='Version:') if vers else None
    type_dd = widgets.Dropdown(options=['All'] + types, description='Issue Type:') if types else None

    def filter_queue(change=None):
        df = queue_df.copy()
        if fam_dd and fam_dd.value != 'All':
            df = df[df['family_id'] == fam_dd.value]
        if ver_dd and ver_dd.value != 'All':
            df = df[df['version'] == ver_dd.value]
        if type_dd and type_dd.value != 'All':
            df = df[df['type'] == type_dd.value]
        clear_output(wait=True)
        display(fam_dd, ver_dd, type_dd)
        display(Markdown(f'### Filtered Feedback Queue ({len(df)})'))
        display(df)

    if fam_dd: fam_dd.observe(filter_queue, names='value')
    if ver_dd: ver_dd.observe(filter_queue, names='value')
    if type_dd: type_dd.observe(filter_queue, names='value')

    display(fam_dd, ver_dd, type_dd)
    filter_queue()
else:
    display(Markdown('No feedback queue data available for filtering.'))

## 7. Ranking Backtest and Threshold Tuning

This section loads the review outcome log, joins it to the feedback queue, and evaluates ranking quality. It tests alternative weight sets, shows best-performing candidates, and recommends threshold cut points from observed outcomes.

In [ ]:
# Ranking backtest and threshold tuning
import numpy as np
import itertools

# Load review log
import pandas as pd
review_log_path = 'reports/model_feedback_review_log.csv'
review_df = pd.read_csv(review_log_path) if Path(review_log_path).exists() else pd.DataFrame()

# Join review log to queue
if not review_df.empty and not queue_df.empty:
    merged = pd.merge(review_df, queue_df, left_on=['target','prediction_family_id','model_version','issue_type'], right_on=['target','family_id','version','type'], how='left', suffixes=('_review','_queue'))
    display(Markdown('### Review Log Joined to Queue'))
    display(merged.head(10))
    # Compare accepted vs rejected by score
    accepted = merged[merged['review_decision']=='accept']
    rejected = merged[merged['review_decision']=='reject']
    display(Markdown(f"Accepted median score: {accepted['composite_priority_score'].median() if not accepted.empty else 'N/A'}"))
    display(Markdown(f"Rejected median score: {rejected['composite_priority_score'].median() if not rejected.empty else 'N/A'}"))
    # Precision@5, @10
    merged_sorted = merged.sort_values('composite_priority_score', ascending=False)
    for k in [5,10]:
        topk = merged_sorted.head(k)
        prec = (topk['review_decision']=='accept').mean() if not topk.empty else np.nan
        display(Markdown(f'Precision@{k}: {prec:.2f}' if not np.isnan(prec) else f'Precision@{k}: N/A'))
    # Try alternative weights
    def score_row(row, w):
        return w[0]*row['severity_component'] + w[1]*row['recurrence_component'] + w[2]*row['affected_count_component'] + w[3]*row['metric_gap_component']
    weight_grid = [np.arange(0.1,0.51,0.05)]*4
    best_prec5 = 0
    best_weights = None
    for w in itertools.product(*weight_grid):
        if abs(sum(w)-1.0)>1e-6: continue
        merged['alt_score'] = merged.apply(lambda r: score_row(r,w), axis=1)
        merged_sorted = merged.sort_values('alt_score', ascending=False)
        top5 = merged_sorted.head(5)
        prec5 = (top5['review_decision']=='accept').mean() if not top5.empty else 0
        if prec5 > best_prec5:
            best_prec5 = prec5
            best_weights = w
    display(Markdown(f'Best Precision@5: {best_prec5:.2f} with weights {best_weights}'))
    # Recommend threshold cut points
    if not merged.empty:
        quantiles = merged['composite_priority_score'].quantile([0.9,0.7,0.5,0.3]).to_dict()
        display(Markdown(f"Suggested threshold bands (90/70/50/30%): {quantiles}"))
else:
    display(Markdown('No review log or queue data available for ranking backtest.'))

## 8. Review Quality Diagnostics

This section flags:
- False positives with high scores (accepted = 0, confirmed = 0, but high priority)
- Confirmed issues with low scores (accepted = 1, confirmed = 1, but low priority)
- Most misweighted component patterns (e.g., high severity but low confirmation)

Use this to diagnose if severity, recurrence, affected count, or metric gap are misweighted or if thresholds are too loose.

In [ ]:
# Review quality diagnostics
if not review_df.empty:
    # False positives: high score, not accepted/confirmed
    high_score_thresh = review_df['composite_priority_score'].quantile(0.8) if not review_df.empty else 0.8
    false_positives = review_df[(review_df['composite_priority_score'] >= high_score_thresh) & (review_df['review_decision'] != 'accept') & (review_df['issue_confirmed'] == 0)]
    display(Markdown(f'### False Positives with High Scores ({len(false_positives)})'))
    display(false_positives.head(5))
    # Confirmed issues with low scores
    low_score_thresh = review_df['composite_priority_score'].quantile(0.2) if not review_df.empty else 0.2
    missed_issues = review_df[(review_df['composite_priority_score'] <= low_score_thresh) & (review_df['review_decision'] == 'accept') & (review_df['issue_confirmed'] == 1)]
    display(Markdown(f'### Confirmed Issues with Low Scores ({len(missed_issues)})'))
    display(missed_issues.head(5))
    # Misweighted component patterns
    comp_cols = ['severity_component','recurrence_component','affected_count_component','metric_gap_component']
    if all(c in review_df.columns for c in comp_cols):
        for c in comp_cols:
            high = review_df[review_df[c] >= 0.8]
            low_conf = high[high['issue_confirmed'] == 0]
            display(Markdown(f'#### High {c} but Not Confirmed ({len(low_conf)})'))
            display(low_conf.head(3))
else:
    display(Markdown('No review data available for diagnostics.'))

## 9. Review Entry Helper

Use this form to log review outcomes for selected queue items. This ensures fast, consistent, and high-quality review capture. All entries are appended to `model_feedback_review_log.csv` with a timestamp and controlled values.

In [ ]:
# Review entry helper
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime
import csv

if not queue_df.empty:
    # Dropdown to select queue item by index
    idx_dd = widgets.Dropdown(options=[(f"{i}: {row['type']} | {row.get('family_id','')} | {row.get('version','')}", i) for i, row in queue_df.iterrows()], description='Queue Item:')
    # Controlled value widgets
    review_decision_dd = widgets.Dropdown(options=['accept','defer','reject'], description='Decision:')
    issue_confirmed_dd = widgets.Dropdown(options=[('Yes',1),('No',0)], description='Confirmed:')
    action_taken_dd = widgets.Dropdown(options=[('Yes',1),('No',0)], description='Action Taken:')
    impact_level_dd = widgets.Dropdown(options=['high','medium','low'], description='Impact:')
    notes_box = widgets.Textarea(description='Notes:', layout=widgets.Layout(width='50%'))
    submit_btn = widgets.Button(description='Log Review', button_style='success')
    out = widgets.Output()

    def on_submit(b):
        with out:
            clear_output()
            idx = idx_dd.value
            row = queue_df.iloc[idx]
            now = datetime.now().isoformat(timespec='seconds')
            entry = [
                now,
                row.get('target',''),
                row.get('family_id',''),
                row.get('version',''),
                row.get('type',''),
                row.get('composite_priority_score',''),
                row.get('severity_component',''),
                row.get('recurrence_component',''),
                row.get('affected_count_component',''),
                row.get('metric_gap_component',''),
                review_decision_dd.value,
                issue_confirmed_dd.value,
                action_taken_dd.value,
                impact_level_dd.value,
                notes_box.value.replace('\n',' ')
            ]
            log_path = 'reports/model_feedback_review_log.csv'
            with open(log_path, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(entry)
            print(f'Review logged for queue item {idx}.')
    submit_btn.on_click(on_submit)
    display(idx_dd, review_decision_dd, issue_confirmed_dd, action_taken_dd, impact_level_dd, notes_box, submit_btn, out)
else:
    display(Markdown('No queue data available for review entry.'))

## 10. Priority-Band Diagnostics

This section analyzes review outcomes by composite priority score bands:
- 0.80–1.00: high priority
- 0.60–0.79: review next
- 0.40–0.59: monitor
- <0.40: backlog/noise

For each band, it shows review count, accept rate, confirm rate, action-taken rate, false-positive rate, and average component values. Charts visualize accept, confirm, and false-positive rates by band.

In [ ]:
# Priority-band diagnostics
import matplotlib.pyplot as plt

if not review_df.empty:
    # Define bands
    def band(score):
        if score >= 0.8: return '0.80–1.00 High'
        if score >= 0.6: return '0.60–0.79 Review Next'
        if score >= 0.4: return '0.40–0.59 Monitor'
        return '<0.40 Backlog'
    review_df['score_band'] = review_df['composite_priority_score'].apply(band)
    bands = ['0.80–1.00 High','0.60–0.79 Review Next','0.40–0.59 Monitor','<0.40 Backlog']
    band_stats = []
    for b in bands:
        sub = review_df[review_df['score_band']==b]
        if sub.empty:
            band_stats.append({'band':b,'count':0,'accept_rate':None,'confirm_rate':None,'action_rate':None,'false_pos_rate':None,'avg_severity':None,'avg_recurrence':None,'avg_affected':None,'avg_metric_gap':None})
            continue
        count = len(sub)
        accept_rate = (sub['review_decision']=='accept').mean()
        confirm_rate = (sub['issue_confirmed']==1).mean()
        action_rate = (sub['action_taken']==1).mean()
        false_pos_rate = ((sub['review_decision']!='accept') & (sub['issue_confirmed']==0)).mean()
        avg_severity = sub['severity_component'].mean()
        avg_recurrence = sub['recurrence_component'].mean()
        avg_affected = sub['affected_count_component'].mean()
        avg_metric_gap = sub['metric_gap_component'].mean()
        band_stats.append({'band':b,'count':count,'accept_rate':accept_rate,'confirm_rate':confirm_rate,'action_rate':action_rate,'false_pos_rate':false_pos_rate,'avg_severity':avg_severity,'avg_recurrence':avg_recurrence,'avg_affected':avg_affected,'avg_metric_gap':avg_metric_gap})
    band_df = pd.DataFrame(band_stats)
    display(band_df)
    # Charts
    fig, axs = plt.subplots(2,2,figsize=(12,8))
    band_df.plot.bar(x='band',y='accept_rate',ax=axs[0,0],legend=False,title='Accept Rate by Band',color='green')
    band_df.plot.bar(x='band',y='confirm_rate',ax=axs[0,1],legend=False,title='Confirm Rate by Band',color='blue')
    band_df.plot.bar(x='band',y='false_pos_rate',ax=axs[1,0],legend=False,title='False-Positive Rate by Band',color='red')
    band_df.plot.bar(x='band',y='count',ax=axs[1,1],legend=False,title='Review Count by Band',color='gray')
    plt.tight_layout()
    plt.show()
else:
    display(Markdown('No review data available for priority-band diagnostics.'))

## 11. Component Drift Over Time

This section tracks, by dated snapshot:
- Average severity, recurrence, affected count, metric gap, and composite priority score
- Accept and confirm rates over time
- Top recurring issue types by period

Charts and tables help interpret whether the queue is getting cleaner, noisier, or shifting in component emphasis.

In [ ]:
# Component drift over time
if not review_df.empty and 'review_timestamp' in review_df.columns:
    # Parse date
    review_df['date'] = pd.to_datetime(review_df['review_timestamp']).dt.date
    # Aggregate by date
    agg = review_df.groupby('date').agg({
        'severity_component':'mean',
        'recurrence_component':'mean',
        'affected_count_component':'mean',
        'metric_gap_component':'mean',
        'composite_priority_score':'mean',
        'review_decision': lambda x: (x=='accept').mean(),
        'issue_confirmed': 'mean'
    }).rename(columns={'review_decision':'accept_rate','issue_confirmed':'confirm_rate'})
    display(agg)
    # Line charts for each component
    ax = agg[['severity_component','recurrence_component','affected_count_component','metric_gap_component','composite_priority_score']].plot(figsize=(12,6), title='Component Averages Over Time')
    ax.set_ylabel('Average Value')
    plt.show()
    # Accept/confirm rates
    ax2 = agg[['accept_rate','confirm_rate']].plot(figsize=(12,4), title='Accept/Confirm Rates Over Time')
    ax2.set_ylabel('Rate')
    plt.show()
    # Top recurring issue types by period
    top_types = review_df.groupby(['date','issue_type']).size().reset_index(name='count')
    pivot = top_types.pivot(index='date', columns='issue_type', values='count').fillna(0)
    pivot.plot(kind='bar', stacked=True, figsize=(14,6), title='Top Recurring Issue Types by Date')
    plt.ylabel('Count')
    plt.show()
    # Biggest upward/downward movers
    if len(agg) > 1:
        movers = agg.diff().iloc[1:]
        display(Markdown('### Biggest Upward Movers'))
        display(movers.max().sort_values(ascending=False))
        display(Markdown('### Biggest Downward Movers'))
        display(movers.min().sort_values())
else:
    display(Markdown('No review data with timestamps available for drift analysis.'))

## 12. Recurring Issue-Type Trend Comparison

This section tracks, by period and issue type:
- Frequency
- Accept rate
- Confirm rate
- Action-taken rate
- Average composite score
- Biggest rising and falling issue types

Charts and tables help distinguish persistent calibration drift from transient noise and identify high-ranking but low-action issue categories.

In [ ]:
# Recurring issue-type trend comparison
if not review_df.empty and 'review_timestamp' in review_df.columns:
    review_df['date'] = pd.to_datetime(review_df['review_timestamp']).dt.date
    # Aggregate by date and issue type
    issue_agg = review_df.groupby(['date','issue_type']).agg({
        'review_decision': lambda x: (x=='accept').mean(),
        'issue_confirmed': 'mean',
        'action_taken': 'mean',
        'composite_priority_score': 'mean',
        'target': 'count'
    }).rename(columns={'review_decision':'accept_rate','issue_confirmed':'confirm_rate','action_taken':'action_rate','target':'count'}).reset_index()
    # Stacked trend chart
    pivot = issue_agg.pivot(index='date', columns='issue_type', values='count').fillna(0)
    pivot.plot(kind='bar', stacked=True, figsize=(14,6), title='Issue Type Frequency Over Time')
    plt.ylabel('Count')
    plt.show()
    # Issue-type quality table
    latest = issue_agg[issue_agg['date']==issue_agg['date'].max()]
    display(latest.sort_values(['confirm_rate','action_rate'], ascending=False))
    # Rising/falling issue types
    if len(issue_agg['date'].unique()) > 1:
        first = issue_agg[issue_agg['date']==issue_agg['date'].min()].set_index('issue_type')
        last = latest.set_index('issue_type')
        rising = (last['count'] - first['count']).sort_values(ascending=False)
        falling = (first['count'] - last['count']).sort_values(ascending=False)
        display(Markdown('### Top Rising Issue Types'))
        display(rising.head(5))
        display(Markdown('### Top Falling Issue Types'))
        display(falling.head(5))
else:
    display(Markdown('No review data with timestamps available for issue-type trend analysis.'))